In [ ]:
# 1. Könyvtárak importálása
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 2. Fájlbeolvasás és alap statisztikák
df1 = pd.read_csv('../data/raw/CPU_benchmark_v4.csv')
print(f"Az adathalmaz teljes mérete: {df1.shape[0]} sor és {df1.shape[1]} oszlop")
print(f"Sorok száma hiányzó értékkel: {df1.isnull().any(axis=1).sum()}")

def get_stats(df, name):
    # Hiányzó értékek (NaN)
    null_counts = df.isnull().sum()
    # Nulla értékek (csak numerikus oszlopoknál releváns)
    zero_counts = (df == 0).sum()

    stats = pd.DataFrame({
        'Hiányzó (NaN)': null_counts,
        'Nulla érték (0)': zero_counts
    })
    print(f"\n--- {name} statisztikák ---")
    print(stats)
    return stats

# Megjelen#t3s
stats1 = get_stats(df1, "CPU_benchmark_v4")

# 3. GLOBÁLIS ADATTISZTÍTÁS (Csak egyszer fut le!)
def clean_numeric(val):
    if isinstance(val, str):
        return float(val.replace(',', ''))
    return val

# A powerPerf, price, stb. konverziója stringből floattá a teljes df1-en
numeric_cols = ['price', 'cpuMark', 'threadMark', 'TDP', 'powerPerf', 'cores']
for col in numeric_cols:
    if col in df1.columns:
        df1[col] = df1[col].apply(clean_numeric)

# 4. GLOBÁLIS VÁLTOZÓK (Socket csoportosítás)
top_20_sockets = df1['socket'].value_counts().nlargest(20).index
df1['socket_grouped'] = df1['socket'].apply(lambda x: x if x in top_20_sockets else 'Other')

# 5. FEATURE LISTÁK
# A) Korrelációs analízishez (itt vizsgálni akarjuk a powerPerf-et is)
corr_features = ['price', 'cpuMark', 'threadMark', 'TDP', 'powerPerf', 'cores', 'testDate', 'category']

# B) Ár (price) modellezéséhez és pótlásához (itt a célváltozók és deriváltak nincsenek bent)
features_price = ['cpuMark', 'threadMark', 'TDP', 'cores', 'testDate', 'category', 'socket_grouped']

# C) Végső Power Performance predikcióhoz (itt az ár már fontos bemeneti változó)
features_pp = ['price', 'cpuMark', 'threadMark', 'TDP', 'cores', 'testDate', 'category', 'socket_grouped']

In [ ]:
# Korreláció-analízis
# Csak azokat az oszlopokat tartjuk meg, amik ténylegesen benne vannak a df1-ben
existing_features = [f for f in corr_features if f in df1.columns]
df_analysis_clean = df1[existing_features].copy()

# One-Hot Encoding alkalmazása
df_corr_analysis = pd.get_dummies(df_analysis_clean, columns=['category'] if 'category' in df_analysis_clean.columns else [], prefix='cat')

# Csak a numerikus oszlopok kiválasztása
numeric_df = df_corr_analysis.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

# Biztonságos kiírás: PowerPerf és TDP közötti korreláció
if 'TDP' in corr_matrix.columns and 'powerPerf' in corr_matrix.columns:
    tdp_power_corr = corr_matrix.loc['TDP', 'powerPerf']
    print(f"--- TDP és PowerPerf közötti korreláció: {tdp_power_corr:.4f} ---\n")

# Price korrelációk megjelenítése
if 'price' in corr_matrix.columns:
    price_corr = corr_matrix['price'].sort_values(ascending=False)
    print('--- Price korrelációk ---')
    print(price_corr)

    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Feature-ök korrelációja')
    plt.tight_layout()
    plt.show()

In [ ]:
# Csak azokat a sorokat nézzük, ahol van ár
df_price_analysis = df1.dropna(subset=['price']).copy()

# 1. Ár eloszlása Category szerint
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_price_analysis, x='category', y='price', palette='Set2')
plt.title('Ár eloszlása kategóriánként')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Átlagos árak kategóriánként:")
display(df_price_analysis.groupby('category')['price'].mean().sort_values(ascending=False))

# 2. Ár eloszlása a leggyakoribb Socket-ek szerint (Top 15)
top_sockets = df_price_analysis['socket'].value_counts().nlargest(15).index
df_top_sockets_price = df_price_analysis[df_price_analysis['socket'].isin(top_sockets)]

plt.figure(figsize=(15, 7))
sns.boxplot(data=df_top_sockets_price, x='socket', y='price', palette='viridis')
plt.title('Ár eloszlása a 15 leggyakoribb foglalat típusnál')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Átlagos árak a leggyakoribb socket-eknél:")
display(df_top_sockets_price.groupby('socket')['price'].mean().sort_values(ascending=False))

In [ ]:
# Finding best Price regression
# Adatok előkészítése: itt a df1 már a tisztított formátumú
df_price_base = df1.dropna(subset=["price", "cpuMark", "threadMark", "TDP", "cores", "testDate", "category"]).copy()

results_list_price = []

# --- 1. Modell eredmények (eredeti adatokon + Socket) ---
# Itt már nyugodtan használhatjuk a fenti 'features_price' listát
y_orig_price = df_price_base["price"]
X_orig_price = pd.get_dummies(df_price_base[features_price], columns=["category", "socket_grouped"])
X_train_orig_price, X_test_orig_price, y_train_orig_price, y_test_orig_price = train_test_split(X_orig_price, y_orig_price, test_size=0.2, random_state=42)

models_orig_price = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_Original": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("--- Modell eredmények (eredeti adatokon - Price) ---")
for name, model in models_orig_price.items():
    model.fit(X_train_orig_price, y_train_orig_price)
    y_pred = model.predict(X_test_orig_price)
    rmse = np.sqrt(mean_squared_error(y_test_orig_price, y_pred))
    mae = mean_absolute_error(y_test_orig_price, y_pred)
    r2 = r2_score(y_test_orig_price, y_pred)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 2. Modell eredmények (logaritmikus transzformációval + Socket) ---
y_orig_log_price = np.log1p(y_orig_price)
X_train_orig_log_price, X_test_orig_log_price, y_train_orig_log_price, y_test_orig_log_price = train_test_split(X_orig_price, y_orig_log_price, test_size=0.2, random_state=42)

models_log_orig_price = {
    "Linear_LogTransformed": LinearRegression(),
    "Ridge_LogTransformed": Ridge(alpha=10.0),
    "Lasso_LogTransformed": Lasso(alpha=0.1),
    "ElasticNet_LogTransformed": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_LogTransformed": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("\n--- Modell eredmények (log transzformáció - Price) ---")
for name, model in models_log_orig_price.items():
    model.fit(X_train_orig_log_price, y_train_orig_log_price)
    y_pred_log = model.predict(X_test_orig_log_price)
    y_pred = np.expm1(y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_test_orig_price, y_pred))
    mae = mean_absolute_error(y_test_orig_price, y_pred)
    r2 = r2_score(y_test_orig_price, y_pred)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 3. Modell eredmények (outlierek kezelése IQR-rel) ---
Q1_price = df_price_base['price'].quantile(0.25)
Q3_price = df_price_base['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price

df_filtered_price = df_price_base[(df_price_base['price'] >= lower_bound_price) & (df_price_base['price'] <= upper_bound_price)].copy()

y_filt_price = df_filtered_price["price"]
X_filt_price = pd.get_dummies(df_filtered_price[features_price], columns=["category", "socket_grouped"])
X_train_filt_price, X_test_filt_price, y_train_filt_price, y_test_filt_price = train_test_split(X_filt_price, y_filt_price, test_size=0.2, random_state=42)

models_filtered_price = {
    "Linear_IQR": LinearRegression(),
    "RandomForest_IQR": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("\n--- Modell eredmények (outlierek kezelése IQR-rel - Price) ---")
for name, model in models_filtered_price.items():
    model.fit(X_train_filt_price, y_train_filt_price)
    y_pred_filt = model.predict(X_test_filt_price)
    rmse_filt = np.sqrt(mean_squared_error(y_test_filt_price, y_pred_filt))
    mae_filt = mean_absolute_error(y_test_filt_price, y_pred_filt)
    r2_filt = r2_score(y_test_filt_price, y_pred_filt)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse_filt, 'MAE': mae_filt, 'R2 Score': r2_filt})

comparison_df_price = pd.DataFrame(results_list_price).sort_values(by="R2 Score", ascending=False)
display(comparison_df_price)

In [ ]:
# Price regression imputation
df_train = df1.dropna(subset=['TDP']).copy()
df_price_base = df_train.dropna(subset=['price'] + features_price).copy()

# Outlier szűrés a legjobb tanító adathalmazért
Q1_p = df_price_base['price'].quantile(0.25)
Q3_p = df_price_base['price'].quantile(0.75)
IQR_p = Q3_p - Q1_p
df_f = df_price_base[(df_price_base['price'] >= (Q1_p - 1.5 * IQR_p)) & (df_price_base['price'] <= (Q3_p + 1.5 * IQR_p))].copy()

y_f = df_f['price']
X_f = pd.get_dummies(df_f[features_price], columns=['category', 'socket_grouped'])

# Modell betanítása a teljes szűrt adathalmazon
best_rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_rf_model.fit(X_f, y_f)

# Hiányzó árak pótlása
df_imputed = df_train.copy()
missing_price_mask = df_imputed['price'].isnull()
df_missing_price = df_imputed[missing_price_mask].copy()

if not df_missing_price.empty:
    X_missing = pd.get_dummies(df_missing_price[features_price], columns=['category', 'socket_grouped'])
    X_missing = X_missing.reindex(columns=X_f.columns, fill_value=0)
    X_missing = X_missing.astype(float)

    predicted_prices = best_rf_model.predict(X_missing)
    df_imputed.loc[missing_price_mask, 'price'] = predicted_prices

# Származtatott értékek újraszámítása a pótolt árak alapján
df_imputed['price_is_imputed'] = missing_price_mask.astype(int)
df_imputed['cpuValue'] = df_imputed['cpuMark'] / df_imputed['price']
df_imputed['threadValue'] = df_imputed['threadMark'] / df_imputed['price']

df_imputed.to_csv('../data/processed/CPU_benchmark_v4_price.csv', index=False)
print('Kiegészített adathalmaz elmentve!')

In [ ]:
# Power Performance prediction (main task)
# Itt már a features_pp globális listát használjuk!
df_pp_train = df_imputed.dropna(subset=['powerPerf']).copy()

X_pp = pd.get_dummies(df_pp_train[features_pp], columns=['category', 'socket_grouped'])
y_pp = df_pp_train['powerPerf']

# 1. Modell validáció
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pp, y_pp, test_size=0.2, random_state=42)

pp_model = RandomForestRegressor(n_estimators=100, random_state=42)
pp_model.fit(X_train_p, y_train_p)

y_pred_p = pp_model.predict(X_test_p)
print(f"Power Performance modell R2 Score: {r2_score(y_test_p, y_pred_p):.4f}")
print(f"Power Performance modell MAE: {mean_absolute_error(y_test_p, y_pred_p):.4f}")

# 2. Hiányzó Power Performance értékek pótlása
df_final = df_imputed.copy()
missing_pp_mask = df_final['powerPerf'].isnull()

if missing_pp_mask.any():
    X_missing_pp = pd.get_dummies(df_final[missing_pp_mask][features_pp], columns=['category', 'socket_grouped'])
    X_missing_pp = X_missing_pp.reindex(columns=X_pp.columns, fill_value=0)

    predicted_pp = pp_model.predict(X_missing_pp)
    df_final.loc[missing_pp_mask, 'powerPerf'] = predicted_pp
    df_final['powerPerf_is_imputed'] = missing_pp_mask.astype(int)

print(f"\nHiányzó Power Performance értékek pótolva: {missing_pp_mask.sum()} sorban.")
df_final.to_csv('../data/processed/CPU_benchmark_final_imputed.csv', index=False)

In [53]:
import joblib

# 1. Mentjük a betanított Power Performance modellt
joblib.dump(pp_model, '../models/powerPerf_rf_model.pkl')

# 2. Mentjük a bemeneti oszlopok listáját (Kritikus a webapphoz!)
joblib.dump(list(X_pp.columns), '../models/model_columns.pkl')

print("Modell és oszlopstruktúra sikeresen becsomagolva és elmentve!")

Modell és oszlopstruktúra sikeresen becsomagolva és elmentve!
